# 실습 4B. 기계독해 (2) — T5로 **답을 써낸다**

**AI아카데미 [A4021] 언어지능: 언어모델 기반 자연어처리 실습 기초 · 2일차 오전**

> **여기부터는 강사 시연입니다.** 학생은 셀을 그대로 실행하면서 화면을 보세요.

앞 노트북(실습 4A)에서 BERT는 지문에서 **답의 자리를 골랐습니다**.
`paust/pko-t5-base` 는 다릅니다 — 지문과 질문을 읽고 답을 **글자로 써냅니다**.

| | **BERT** (실습 4A) | **pko-T5** (이 노트북) |
|---|---|---|
| 출력 | 토큰마다 시작·끝 점수 두 줄 | 토큰을 하나씩 이어 붙인 문장 |
| 답의 정체 | 지문에서 잘라낸 구간 | 새로 쓴 글자열 |
| 지문에 없는 말 | **못 낸다** | 낼 수 있다 |
| 형식이 깨질 수 있나 | 없다 | **있다** |
| 태스크를 바꾸려면 | head를 바꾼다 | 지시문을 바꾼다 |

**같은 학습셋, 같은 평가셋 300건, 같은 채점 함수**를 씁니다. 그래야 두 방식을 나란히 놓을 수 있습니다.
마지막에 실습 4A의 결과를 불러와 비교표를 만듭니다 — **실습 4A를 먼저 돌려 두세요.**

## 순서와 소요시간

| | 내용 | 시간 |
|---|---|---|
| 1~3 | 환경 확인 · 데이터와 지시문 보기 | 5분 |
| 4~5 | T5 학습 · 평가 | 20분 |
| 6~7 | 맞힌 예·틀린 예 보기 · BERT와 비교 | 15분 |

## 1. 환경 확인

저장소 **루트**를 작업 폴더로 삼습니다. 공통 모듈은 1일차와 같은 `day1/lab_common.py` 입니다.

In [ ]:
import os, sys, json, time
from pathlib import Path

# 노트북이 day2/ 또는 day2/instructor/ 에 있어도 저장소 루트를 찾아 이동한다.
# 공통 모듈은 1일차에 쓰던 day1/lab_common.py 를 그대로 쓴다.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "day1" / "lab_common.py").exists()), None)
assert _root is not None, "DeepKNLP 저장소 안에서 이 노트북을 여세요"
os.chdir(_root); sys.path.insert(0, str(_root / "day1"))
PY = sys.executable                       # 셸 명령(!)에서 이 커널과 같은 파이썬을 쓰기 위해

import lab_common as L
from datasets import Dataset

from common import TASKS

TASK = "mrc"
MODEL = L.MODEL_T5
OUT = L.OUT_DIR_DAY2                      # 결과는 output/day2/ 에 쌓인다
T5_EPOCHS, T5_LR, T5_BATCH = 3, 3e-4, 4   # T5 학습 설정 (배치 4 — 아래 설명)

L.env_info()

## 2. 데이터 보기 — KorQuAD

학습 **800건**, 평가 **300건**입니다. 1일차와 같은 방식으로, 2일차 오후에 LLM에게 줄 것과
**똑같은 예제들**을 씁니다. 그래야 마지막에 세 방식을 나란히 비교할 수 있습니다.

각 예제는 **지문(context) · 질문(question) · 정답(answer)** 세 부분입니다.
정답에는 글자 위치(`answer_start`)가 함께 붙어 있습니다 — 이것이 미션에서 쓸 재료입니다.

In [ ]:
train_rows = L.load_budget(TASK)
eval_rows = L.load_eval(TASK)
L.check_no_leak(train_rows, eval_rows)

_r = train_rows[0]
print("\n[예제 하나 뜯어보기]")
print("지문 :", _r["input"]["context"][:120], "…")
print("질문 :", _r["input"]["question"])
print("정답 :", _r["raw"]["answer_text"][0], f'(지문의 {_r["raw"]["answer_start"][0]}번째 글자부터)')
print("확인 :", repr(_r["input"]["context"][_r["raw"]["answer_start"][0]:][:len(_r["raw"]["answer_text"][0])]))

In [ ]:
print("[평가셋 예시 3건]")
L.preview(eval_rows, 3)

## 3. T5가 받는 것 — 지시문

T5에게는 head가 없습니다. 대신 **지시문**을 글로 적어 줍니다.
아래 지시문은 2일차 오후에 LLM에게 줄 것과 **한 글자도 다르지 않습니다**
(`task5-llm-ft/common.py` 의 `TASKS["mrc"]` 한 곳에 모아 두었습니다).
같은 문장을 주어야 세 방식을 공정하게 비교할 수 있습니다.

In [ ]:
t5_tok, t5_model = L.load_t5(MODEL)
_src, _tgt = L.to_text(train_rows[:1])
print("─ T5가 받는 입력 ──────────────────────────────")
print(_src[0][:600])
print("\n─ T5가 써내야 하는 답 ─────────────────────────")
print(_tgt[0])
print("───────────────────────────────────────────────")

### 미션 `m-t5mrc-1` · `m-t5mrc-2` · `m-t5mrc-3` — 학습 전 T5 에게 한 건 물어보기

`L.generate_t5` 가 평가셋 300건에 대해 하는 일을, **한 건만** 손으로 해 봅니다.
그래야 "생성"이 무엇을 정해 주어야 하는 일인지 보입니다. 이 셀에서 채울 것은 **`____` 두 곳**이고,
세 번째(`m-t5mrc-3`)는 바로 아래 미션 셀 ②에 있습니다.

| | 정할 것 | 문항 | 어디 |
|---|---|---|---|
| ① | 얼마나 길게 써내게 할까 | **`m-t5mrc-1`** | 미션 셀 ① |
| ② | 특수 토큰을 답에 남길까 | **`m-t5mrc-2`** | 미션 셀 ① |
| ③ | 지문에 근거한 답인지 **무엇과 비교할까** | **`m-t5mrc-3`** | 미션 셀 ② (바로 아래) |

아래 객관식 두 개를 먼저 풀어 보세요. 고른 답을 그 다음 셀의 `____` 에 그대로 옮기면 됩니다.

In [ ]:
L.quiz("m-t5mrc-1")

In [ ]:
L.quiz("m-t5mrc-2")

In [ ]:
# 아래 ____ 두 곳만 채우세요. 바로 위 셀의 객관식에서 고른 답을 그대로 옮기면 됩니다.

# 학습 전 T5 에게 한 건만 물어본다 — 나중에 학습 후와 비교하기 위한 기준선
_src1, _tgt1 = L.to_text(eval_rows[:1])
_batch1 = t5_tok(_src1, return_tensors="pt", truncation=True,
                 max_length=L.MAX_SOURCE_T5).to(t5_model.device)
_out1 = t5_model.generate(**_batch1,
                          max_new_tokens=____,                              # m-t5mrc-1
                          num_beams=1, do_sample=False)
_ans1 = t5_tok.batch_decode(_out1, skip_special_tokens=____)[0]   # m-t5mrc-2
print("정답      :", _tgt1[0])
print("T5 답     :", repr(_ans1[:80]))          # 줄바꿈까지 보이게 repr 로 (학습 전에는 빈 줄만 나오기도 한다)
print("(학습 전이라 엉뚱하거나 비어 있어도 정상입니다 — 이것이 기준선입니다)")

### 미션 셀 ② — `m-t5mrc-3` · 지문에 근거한 답인가

4B 의 요점은 **생성형은 지문에 없는 말을 써낼 수 있다**는 것입니다. 추출형은 구조상 그럴 수 없습니다.
그것을 말로만 하지 않고 **숫자로** 보려고 합니다 — 답 하나하나가 지문에 글자 그대로 들어 있는지 세는 함수입니다.
채울 것은 **`____` 한 곳**, 무엇과 비교할 것인가입니다.

이 함수는 §6 에서 300건 전체에 쓰입니다.

In [ ]:
L.quiz("m-t5mrc-3")

In [ ]:
# ____ 한 곳만 채우세요. 위 카드에서 고른 답을 그대로 옮기면 됩니다.

def grounded(pred, context):
    """이 답이 **지문에 글자 그대로** 들어 있는가.

    추출형(BERT)은 지문에서 잘라내므로 언제나 True 다.
    생성형(T5)은 답을 써내므로 False 가 나올 수 있다 — 그것이 4B 의 요점이다.
    맞았는지와는 다른 물음이다. 지문에 없는 말로 정답을 맞힐 수도, 지문에 있는 말로 틀릴 수도 있다.
    """
    pred = pred.strip()
    return bool(pred) and pred in ____   # m-t5mrc-3

In [ ]:
# 확인 — 지문에 있는 답과 없는 답을 하나씩
assert grounded("대전광역시", "ETRI는 대전광역시 유성구에 있다.") is True, "지문에 있는 답인데 False 가 나왔습니다"
assert grounded("서울", "ETRI는 대전광역시 유성구에 있다.") is False, "지문에 없는 답인데 True 가 나왔습니다"
assert grounded("", "아무 지문") is False, "빈 답은 근거 없음으로 세어야 합니다"
print("통과 — 지문에 있는 답과 없는 답을 구분합니다")

## 4. 학습

`task5-llm-ft/t5_baseline.py --task mrc --mode budget` 과 거의 같은 설정입니다 —
epochs 3, 학습률 3e-4. T5는 bf16에서 불안정한 사례가 있어 fp32로 둡니다.

**배치만 8이 아니라 4입니다.** 기계독해는 지문이 들어가 한 칸이 훨씬 크고, 배치 8로 두면
GPU를 약 15GB 씁니다. 24GB 카드에 들어가기는 하지만, 앞 노트북(실습 4A)의 커널을 켜 둔 채로
이 노트북을 열면 둘이 합쳐 위험해집니다. 배치 4면 여유 있게 돕니다 — 시간은 조금 더 걸립니다.

**BERT보다 오래 걸립니다.** 답을 고르는 것이 아니라 글자를 하나씩 만들어 내도록 배워야 하기 때문입니다.

> **`CUDA out of memory` 가 나면** — 앞 노트북(실습 4A)의 커널이 아직 살아 있을 가능성이 큽니다.
> Jupyter 왼쪽의 실행 중인 커널 목록에서 그것을 종료하고 이 노트북의 커널을 다시 시작하세요.
> 그래도 나면 `T5_BATCH` 를 2로 줄이면 됩니다.

In [ ]:
L.set_seed(42)

t5_train_ds = L.encode_t5(t5_tok, train_rows, TASK)
t5_trainer, t5_meta = L.train_t5(t5_model, t5_tok, t5_train_ds, TASK,
                                 epochs=T5_EPOCHS, lr=T5_LR, batch_size=T5_BATCH)

# 학습한 모델을 디스크에 남긴다 — 뒤에서 데모 서버가 이것을 읽는다 (약 1.1GB)
L.save_model(t5_model, t5_tok, "mrc", "t5")

### 퀴즈 `q-t5mrc-2` — 4A 의 전처리가 4B 에는 없는 까닭

4A 에서 그렇게 길었던 전처리가 여기서는 왜 세 줄로 끝났는지 묻는 문제입니다. 보기를 고르면 해설이 열립니다.

In [ ]:
L.quiz("q-t5mrc-2")

## 5. 평가

**BERT와 똑같은 평가셋 300건**입니다. 다만 답을 만드는 방식이 다릅니다 —
BERT는 점수가 가장 높은 두 자리를 한 번에 골랐지만, T5는 글자를 하나씩 이어 붙입니다(생성).

In [ ]:
t5_preds = L.generate_t5(t5_model, t5_tok, eval_rows, TASK)
print("생성 예시 3건:", t5_preds[:3], "\n")

t5_summary = L.evaluate(TASK, t5_preds, eval_rows)
L.save_result(TASK, "t5", t5_summary, t5_meta, out_dir=OUT)

## 6. 맞힌 예와 틀린 예 보기 — 여기서 차이가 드러납니다

T5가 **잘 맞힌 세 건을 먼저, 크게 틀린 세 건을 그다음에** 봅니다.
맞힌 쪽에서는 T5도 지문에서 답을 제대로 찾아낸다는 것이 보이고,
틀린 쪽에서 BERT의 실수(실습 4A §7)와 **종류가 다른지**가 드러납니다.

- BERT는 **지문 안의 엉뚱한 구간**을 짚습니다. 틀려도 지문에 있는 말입니다.
- T5는 지문에 **없는 말을 쓸 수 있습니다.** 그럴듯하지만 지문에 근거가 없는 답이 나오면
  그것이 바로 강의에서 말한 "지어내기"입니다.

In [ ]:
L.examples(TASK, t5_preds, eval_rows, 3)   # 맞힌 것 3건 + 틀린 것 3건

In [ ]:
# T5의 답이 지문 안에 실제로 있는지 세어 본다 — 미션 셀 ②의 grounded 를 300건에 적용. BERT는 100%일 수밖에 없다
_in_ctx = sum(1 for p, r in zip(t5_preds, eval_rows) if grounded(p, r["input"]["context"]))
_empty = sum(1 for p in t5_preds if not p.strip())
print(f"T5의 답 {len(t5_preds)}건 중 지문 안에 그대로 있는 것: {_in_ctx}건 "
      f"({_in_ctx/len(t5_preds)*100:.1f}%)")
print(f"  빈 답: {_empty}건")
print(f"  → 나머지 {len(t5_preds)-_in_ctx}건은 T5가 지문에 없는 형태로 써낸 답입니다.")
print("     BERT(추출형)에서는 이 값이 구조상 100%가 됩니다.")

## 7. 비교 — 같은 데이터, 같은 평가셋, 같은 채점

실습 4A(BERT)의 결과를 불러와 나란히 놓습니다. **실습 4A를 먼저 돌려야** 표가 채워집니다.

숫자보다 **왜 그런지**가 중요합니다.

- 추출형은 답이 지문에 그대로 있는 질문에 강합니다. 형식이 깨질 일이 없고, 근거 구간이 분명합니다.
- 생성형은 표현을 바꿔 쓰거나 종합해야 하는 질문도 시도할 수 있습니다. 대신 지어낼 수 있습니다.
- KorQuAD 는 **정답이 지문에 그대로 있도록 만든 데이터**입니다. 그러니 이 비교는
  추출형에게 유리한 운동장입니다. 그 점을 감안하고 표를 읽으세요.

In [ ]:
bert_summary = L.load_result(TASK, "bert", out_dir=OUT)
if bert_summary is None:
    print("실습 4A(04_기계독해_BERT.ipynb)를 먼저 실행하세요. BERT 결과가 없어 T5만 표시합니다.")
    import pandas as pd
    display(pd.DataFrame([{"방식": "T5 (인코더-디코더)", "EM": t5_summary["em"], "F1": t5_summary["f1"]}]))
else:
    L.compare(TASK, bert_summary, t5_summary)

### 퀴즈 `q-t5mrc-3` — 이 비교표를 읽을 때 감안할 것

방금 본 비교표를 읽을 때 **데이터가 어느 쪽에 맞춰 만들어졌는지**를 감안해야 하는 이유를 묻는 문제입니다. 보기를 고르면 해설이 열립니다.

In [ ]:
L.quiz("q-t5mrc-3")

## 8. 내 모델 데모 — 두 방식에 같은 질문을 넣어 보기

`day2/serve_mrc.py` 가 작은 웹 페이지를 띄웁니다. **`--kind both` 로 띄우면 4A의 BERT와 4B의 T5가
같은 질문에 나란히 답합니다** — 오늘 배운 차이를 한 화면에서 보는 자리입니다.

먼저 GPU를 비우고(아래 셀), **켜기 셀** `L.demo_start("both", port=9006)` 로 띄웁니다 — 배경에서 돌아 셀이 곧 끝나고, **끄기 셀** `L.demo_stop(9006)` 로 내립니다. 몇 번이든 켰다 껐다 할 수 있습니다. 터미널에서 띄워도 됩니다:

```bash
python day2/serve_mrc.py --kind both --port 9006     # 둘을 나란히
python day2/serve_mrc.py --kind t5   --port 9006     # T5만
```

브라우저에서 <http://localhost:9006>. 터미널에서 직접 띄웠다면 멈출 때 `Ctrl+C` 입니다.

In [ ]:
# 데모 서버가 같은 GPU를 쓰므로 노트북이 잡은 메모리를 먼저 풀어 준다
# (나중에 다시 돌아와 이 셀만 실행해도 되게 — 변수가 없으면 조용히 넘어간다)
for _n in ("t5_model", "t5_trainer"):
    globals().pop(_n, None)
L.free_gpu()
import torch
if torch.cuda.is_available():
    print(f"지금 GPU 사용량: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print("아래 켜기 셀을 실행하세요 (또는 터미널에서:  python day2/serve_mrc.py --kind both --port 9006)")

In [ ]:
# ▶ 켜기 — BERT(4A)와 T5(이 노트북)를 나란히. 배경에서 돌므로 이 셀은 곧 끝납니다
L.demo_start("both", port=9006)

In [ ]:
# ■ 끄기 — 다 보고 나면 실행하세요. 다시 보려면 켜기 셀을 다시 실행하면 됩니다
L.demo_stop(9006)

**넣어 볼 것들** — **같은 질문을 4A의 BERT에도 넣어 보세요.** 거기서 오늘의 요점이 드러납니다.

- **답이 지문에 없는 질문.** BERT는 지문 안의 위치만 고르므로 **지어낼 수 없습니다.**
  T5는 **지어냅니다** — 그럴듯한 답이 나오는지, 그것이 왜 위험한지 보세요.
- **지문에 없는 표현으로 묻기.** 지문이 "설립되었다" 인데 "언제 만들어졌나요?" 로.
- **여러 문장을 합쳐야 답이 되는 질문.** 지문 한 군데를 잘라서는 답할 수 없는 것 —
  BERT는 구조상 못 하고 T5는 할 수도 있습니다.
- **숫자를 세어야 하는 질문.** "몇 명인가요?" 처럼 지문에 그 숫자가 적혀 있지 않은 것.

> **관찰 포인트.** T5가 지문에 **없는** 답을 냈을 때, 그것이 맞았나요 틀렸나요?
> 맞았다면 편리하고, 틀렸다면 **확인할 방법이 없습니다** — 추출형은 최소한 "지문 어디서 왔는지"를 가리킵니다.
> 이 교환 관계가 내일 실습5의 LLM에서 그대로 커집니다.

## 9. 퀴즈 `q-t5mrc`

§6 에서 센 것(지문 안에 그대로 있는 답의 비율)이 점수에 어떻게 나타나는지 정리해 보는 문제입니다.
보기를 고르면 해설이 열립니다.

In [ ]:
L.quiz("q-t5mrc")

## 10. 더 해보기

1. **beam search 를 켜면?** `L.generate_t5` 는 `num_beams=1`(greedy)입니다.
   `task5-llm-ft/t5_baseline.py` 에서 값을 올려 보고 EM이 오르는지, 얼마나 느려지는지 재 보세요.
2. **지문에 없는 답을 요구하면?** "이 지문에 나온 기관은 모두 몇 개인가?" 처럼
   **세어야 하는** 질문을 만들어 두 모델에 넣어 보세요. 추출형은 구조상 답할 수 없습니다.
3. **데이터를 늘리면?** 터미널에서:

   ```
   python task5-llm-ft/t5_baseline.py --task mrc --mode full --save output/day2/mrc-t5-full.json
   ```

---

오후에는 세 번째 구조인 **LLM(디코더)** 로 넘어갑니다. 오늘 본 기계독해에
주제분류·문장유사도·개체명인식·SQL생성·수학추론까지 더해 **여섯 태스크를 모델 하나로** 배웁니다.